# Data Visualization: Sleep Health and Lifestyle Dataset

**Mata Kuliah:** Data Mining (Pertemuan 6) — Studi Kasus Data Visualization dengan Python

| | |
|---|---|
| **Nama** | Bonivasius Adiya Dalung |
| **NIM** | 2218041 |
| **Library** | Pandas, NumPy, Matplotlib, Seaborn |

---

## 1. Latar Belakang dan Tujuan

Tidur merupakan salah satu faktor utama yang memengaruhi kesehatan fisik dan mental. Durasi dan kualitas tidur dipengaruhi oleh banyak hal, seperti tingkat stres, aktivitas fisik, pekerjaan, usia, dan kondisi tubuh. Dengan visualisasi data, pola-pola tersebut dapat dikenali lebih cepat daripada hanya membaca tabel angka.

**Tujuan proyek:**
1. Memahami karakteristik dataset (struktur, distribusi, kualitas data).
2. Mengeksplorasi distribusi durasi tidur responden.
3. Membandingkan kualitas tidur antar kelompok (pekerjaan, tingkat stres, kategori BMI, usia).
4. Menganalisis hubungan antarvariabel numerik.
5. Menarik insight dari setiap visualisasi dan menyusun kesimpulan.

## 2. Deskripsi Dataset

Dataset **Sleep Health and Lifestyle** berisi 374 responden dengan 13 variabel mengenai pola tidur dan gaya hidup.

| Variabel | Tipe | Keterangan |
|---|---|---|
| Person ID | Numerik | Identitas responden |
| Gender | Kategorikal | Male / Female |
| Age | Numerik | Usia (tahun) |
| Occupation | Kategorikal | Pekerjaan |
| Sleep Duration | Numerik | Durasi tidur (jam/hari) |
| Quality of Sleep | Numerik (ordinal) | Skor kualitas tidur (skala 1–10) |
| Physical Activity Level | Numerik | Durasi aktivitas fisik (menit/hari) |
| Stress Level | Numerik (ordinal) | Skor tingkat stres (skala 1–10) |
| BMI Category | Kategorikal | Normal / Overweight / Obese |
| Blood Pressure | Teks | Tekanan darah (sistolik/diastolik) |
| Heart Rate | Numerik | Detak jantung istirahat (bpm) |
| Daily Steps | Numerik | Jumlah langkah per hari |
| Sleep Disorder | Kategorikal | None / Insomnia / Sleep Apnea |

## 3. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Pengaturan tampilan visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

print("Library berhasil diimpor.")

## 4. Memuat Dataset

> **Catatan penting:** nilai `None` pada kolom *Sleep Disorder* berarti **responden tidak memiliki gangguan tidur**. Secara default Pandas membaca teks `None` sebagai nilai kosong (NaN), sehingga pembacaan file menggunakan `keep_default_na=False` agar kategori ini tidak salah dianggap sebagai data hilang.

In [ ]:
nama_file = "Sleep_health_and_lifestyle_dataset.csv"

# Mencari file di folder Colab (/content) atau folder kerja saat ini
kandidat = [Path("/content") / nama_file, Path(nama_file)]
path_file = next((p for p in kandidat if p.exists()), None)

# Jika belum ada, minta unggah file (khusus Google Colab)
if path_file is None:
    try:
        from google.colab import files
        print("File belum ditemukan. Silakan unggah file CSV.")
        files.upload()
        path_file = Path(nama_file)
    except ImportError:
        raise FileNotFoundError(f"File {nama_file} tidak ditemukan.")

df = pd.read_csv(path_file, keep_default_na=False, na_values=[""])
df.columns = df.columns.str.strip()

print("Dataset berhasil dibaca!")
print("Jumlah baris :", df.shape[0])
print("Jumlah kolom :", df.shape[1])
n_kolom_asli = df.shape[1]
display(df.head())

## 5. Eksplorasi Awal Data

In [ ]:
print("INFORMASI DATASET")
df.info()

print("\nSTATISTIK DESKRIPTIF (VARIABEL NUMERIK)")
display(df.describe().T.round(2))

print("\nSTATISTIK DESKRIPTIF (VARIABEL KATEGORIKAL)")
display(df.describe(include="object").T)

In [ ]:
print("JUMLAH NILAI KOSONG PER KOLOM")
display(df.isnull().sum().to_frame("Jumlah Nilai Kosong"))

print("\nJUMLAH BARIS DUPLIKAT (seluruh kolom):", df.duplicated().sum())
print("JUMLAH BARIS DUPLIKAT (tanpa Person ID):",
      df.drop(columns="Person ID").duplicated().sum())

print("\nNILAI PADA KOLOM KATEGORIKAL")
for kolom in ["Gender", "BMI Category", "Sleep Disorder"]:
    print(f"\n{kolom}")
    print(df[kolom].value_counts())

**Temuan awal:**
- Tidak ada nilai kosong dan tidak ada baris duplikat jika `Person ID` ikut diperhitungkan.
- Kategori *BMI Category* memuat dua label yang sebenarnya sama, yaitu **Normal** dan **Normal Weight**. Keduanya perlu digabung.
- Jika `Person ID` diabaikan, terdapat 242 baris dengan profil yang identik dengan baris lain. Artinya banyak responden memiliki nilai variabel yang sama persis, sehingga dataset ini cenderung berpola berulang. Hal ini diperhatikan saat menginterpretasi hasil.
- *Sleep Disorder* terdiri dari tiga kategori: None (219), Sleep Apnea (78), dan Insomnia (77).

## 6. Pembersihan dan Persiapan Data

In [ ]:
# 1) Menyamakan label BMI
df["BMI Category"] = df["BMI Category"].replace({"Normal Weight": "Normal"})

# 2) Memecah tekanan darah menjadi sistolik dan diastolik
df[["Sistolik", "Diastolik"]] = (
    df["Blood Pressure"].str.split("/", expand=True).astype(int)
)

# 3) Kolom numerik utama untuk analisis
kolom_numerik = [
    "Age", "Sleep Duration", "Quality of Sleep",
    "Physical Activity Level", "Stress Level",
    "Heart Rate", "Daily Steps"
]
for kolom in kolom_numerik:
    df[kolom] = pd.to_numeric(df[kolom], errors="coerce")

# 4) Kategori tingkat stres
df["Kategori Stres"] = pd.cut(
    df["Stress Level"],
    bins=[-np.inf, 3, 6, np.inf],
    labels=["Rendah (≤3)", "Sedang (4–6)", "Tinggi (≥7)"]
)

# 5) Kelompok usia
urutan_usia = ["27–30", "31–35", "36–40", "41–45", "46–50", "51–59"]
df["Kelompok Usia"] = pd.cut(
    df["Age"], bins=[26, 30, 35, 40, 45, 50, 59], labels=urutan_usia
)

print("BMI Category setelah dibersihkan:")
print(df["BMI Category"].value_counts())
print("\nJumlah responden per kategori stres:")
print(df["Kategori Stres"].value_counts().sort_index())
display(df.head())

## 7. Visualisasi Data

### 7.1 Histogram — Distribusi Durasi Tidur

In [ ]:
data_tidur = df["Sleep Duration"]

plt.figure(figsize=(10, 6))
sns.histplot(data_tidur, bins=15, kde=True, color="#4C72B0")
plt.axvline(data_tidur.mean(), color="red", linestyle="--",
            label=f"Rata-rata = {data_tidur.mean():.2f} jam")
plt.axvline(data_tidur.median(), color="green", linestyle="-.",
            label=f"Median = {data_tidur.median():.2f} jam")
plt.axvline(7, color="black", linestyle=":",
            label="Batas rekomendasi minimal (7 jam)")

plt.title("Distribusi Durasi Tidur Responden")
plt.xlabel("Durasi Tidur (jam per hari)")
plt.ylabel("Jumlah Responden")
plt.legend()
plt.tight_layout()
plt.show()

print("Minimum :", data_tidur.min(), "jam")
print("Maksimum:", data_tidur.max(), "jam")
print("Responden tidur < 7 jam:", (data_tidur < 7).sum(),
      f"({(data_tidur < 7).mean()*100:.1f}%)")

**Interpretasi:**
Durasi tidur responden berkisar antara 5,8 hingga 8,5 jam dengan rata-rata 7,13 jam dan median 7,2 jam. Nilai rata-rata dan median yang berdekatan menunjukkan distribusi yang cukup seimbang. Namun, sebanyak 155 responden (41,4%) tidur kurang dari 7 jam, sehingga hampir separuh responden berada di bawah batas durasi tidur yang umum direkomendasikan bagi orang dewasa. Histogram tampak memiliki beberapa puncak karena durasi tidur tercatat dalam kelipatan 0,1 jam dan banyak responden memiliki nilai yang sama.

### 7.2 Bar Chart — Rata-rata Kualitas Tidur Berdasarkan Pekerjaan

In [ ]:
MIN_RESPONDEN = 10   # pekerjaan dengan responden terlalu sedikit tidak ditampilkan

jumlah = df["Occupation"].value_counts()
pekerjaan_valid = jumlah[jumlah >= MIN_RESPONDEN].index
dikeluarkan = jumlah[jumlah < MIN_RESPONDEN]
print("Pekerjaan dengan n <", MIN_RESPONDEN, "(tidak ditampilkan):")
print(dikeluarkan.to_string(), "\n")

rata_kualitas = (
    df[df["Occupation"].isin(pekerjaan_valid)]
    .groupby("Occupation")["Quality of Sleep"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)

plt.figure(figsize=(11, 6))
ax = sns.barplot(x=rata_kualitas["mean"], y=rata_kualitas.index,
                 hue=rata_kualitas.index, palette="viridis", legend=False)

for i, (rata, n) in enumerate(zip(rata_kualitas["mean"], rata_kualitas["count"])):
    ax.text(rata + 0.05, i, f"{rata:.2f}  (n={n})", va="center",
            bbox=dict(facecolor="white", edgecolor="none", pad=1.5))

plt.axvline(df["Quality of Sleep"].mean(), color="red", linestyle="--",
            label=f"Rata-rata keseluruhan = {df['Quality of Sleep'].mean():.2f}")
plt.title("Rata-rata Kualitas Tidur Berdasarkan Pekerjaan")
plt.xlabel("Rata-rata Skor Kualitas Tidur (skala 1–10)")
plt.ylabel("Pekerjaan")
plt.xlim(0, 10)
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.show()

**Interpretasi:**
Pekerjaan **Engineer** memiliki rata-rata kualitas tidur tertinggi (8,41), diikuti **Lawyer** dan **Accountant** (sekitar 7,89). Sebaliknya, **Salesperson** memiliki rata-rata terendah (6,00), disusul **Doctor** (6,65) dan **Teacher** (6,97). Selisih antara pekerjaan tertinggi dan terendah mencapai sekitar 2,4 poin, sehingga kualitas tidur tampak berbeda cukup jelas antar kelompok pekerjaan. Pekerjaan dengan kurang dari 10 responden (Manager, Sales Representative, Scientist, Software Engineer) tidak ditampilkan karena rata-ratanya tidak cukup representatif. Perbedaan ini kemungkinan terkait faktor lain seperti beban kerja dan tingkat stres, bukan semata-mata jenis pekerjaan.

### 7.3 Boxplot — Durasi Tidur Berdasarkan Kategori Stres

In [ ]:
urutan_stres = ["Rendah (≤3)", "Sedang (4–6)", "Tinggi (≥7)"]

plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="Kategori Stres", y="Sleep Duration",
            order=urutan_stres, hue="Kategori Stres", hue_order=urutan_stres,
            palette="RdYlGn_r", legend=False)
sns.stripplot(data=df, x="Kategori Stres", y="Sleep Duration",
              order=urutan_stres, color="black", alpha=0.15, size=3)

plt.title("Distribusi Durasi Tidur Berdasarkan Kategori Tingkat Stres")
plt.xlabel("Kategori Tingkat Stres")
plt.ylabel("Durasi Tidur (jam per hari)")
plt.tight_layout()
plt.show()

display(
    df.groupby("Kategori Stres", observed=False)["Sleep Duration"]
      .agg(["count", "median", "mean"]).round(2)
)

**Interpretasi:**
Semakin tinggi tingkat stres, semakin pendek durasi tidur. Median durasi tidur kelompok stres **rendah** adalah 8,20 jam, kelompok **sedang** 7,30 jam, dan kelompok **tinggi** hanya 6,15 jam. Perbedaan antar kelompok terlihat besar dan kotak boxplot hampir tidak saling tumpang tindih antara kelompok rendah dan tinggi. Pola ini konsisten dengan korelasi negatif yang kuat antara stres dan durasi tidur pada heatmap (r = −0,81).

### 7.4 Scatter Plot — Hubungan Durasi Tidur dan Kualitas Tidur

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, x="Sleep Duration", y="Quality of Sleep",
    hue="Sleep Disorder", hue_order=["None", "Insomnia", "Sleep Apnea"],
    palette={"None": "#55A868", "Insomnia": "#C44E52", "Sleep Apnea": "#4C72B0"},
    alpha=0.5, s=70
)
sns.regplot(data=df, x="Sleep Duration", y="Quality of Sleep",
            scatter=False, color="black", line_kws={"linestyle": "--"},
            label="Garis regresi")

plt.title("Hubungan Durasi Tidur dan Kualitas Tidur")
plt.xlabel("Durasi Tidur (jam per hari)")
plt.ylabel("Kualitas Tidur (skor 1–10)")
plt.legend(title="Gangguan Tidur")
plt.tight_layout()
plt.show()

korelasi_tidur = df["Sleep Duration"].corr(df["Quality of Sleep"])
print("Korelasi Pearson:", round(korelasi_tidur, 3))

**Interpretasi:**
Titik-titik membentuk pola naik dari kiri bawah ke kanan atas dengan koefisien korelasi Pearson **r = 0,88**, artinya hubungan positif yang sangat kuat: responden yang tidur lebih lama cenderung menilai kualitas tidurnya lebih baik. Responden dengan insomnia (merah) lebih banyak berada di area durasi pendek dan kualitas rendah, sedangkan responden tanpa gangguan (hijau) lebih banyak berada di area kanan atas, meskipun sebagian responden sleep apnea (biru) juga tercatat memiliki kualitas tidur tinggi (skor 9). Karena banyak responden memiliki nilai yang sama, banyak titik saling menumpuk sehingga digunakan transparansi (alpha). Korelasi menunjukkan keterkaitan, bukan bukti sebab-akibat.

### 7.5 Heatmap — Korelasi Antarvariabel Numerik

In [ ]:
korelasi = df[kolom_numerik].corr()
mask = np.triu(np.ones_like(korelasi, dtype=bool), k=1)

plt.figure(figsize=(11, 8))
sns.heatmap(korelasi, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, square=True,
            cbar_kws={"label": "Koefisien Korelasi Pearson"})
plt.title("Heatmap Korelasi Variabel Numerik")
plt.tight_layout()
plt.show()

atas = korelasi.where(np.triu(np.ones(korelasi.shape), k=1).astype(bool))
pasangan = atas.stack().sort_values(key=abs, ascending=False)
print("Enam pasangan dengan korelasi absolut terbesar:")
display(pasangan.head(6).round(3).to_frame("Koefisien Korelasi"))

**Interpretasi:**
Pasangan variabel dengan hubungan terkuat adalah:
- **Kualitas tidur dan stres** (r = −0,90): semakin tinggi stres, semakin rendah kualitas tidur.
- **Durasi tidur dan kualitas tidur** (r = +0,88): tidur lebih lama berkaitan dengan kualitas tidur lebih baik.
- **Durasi tidur dan stres** (r = −0,81): stres tinggi berkaitan dengan tidur lebih singkat.
- **Aktivitas fisik dan langkah harian** (r = +0,77): wajar, karena keduanya mengukur hal yang berdekatan.
- **Stres dan detak jantung** (r = +0,67) serta **kualitas tidur dan detak jantung** (r = −0,66): detak jantung istirahat yang lebih tinggi cenderung muncul bersama stres tinggi dan kualitas tidur rendah.

Stres muncul sebagai variabel yang paling berkaitan dengan kualitas dan durasi tidur dalam dataset ini.

### 7.6 Bar Chart Berkelompok — Gangguan Tidur Berdasarkan Kategori BMI

In [ ]:
urutan_bmi = ["Normal", "Overweight", "Obese"]
urutan_gangguan = ["None", "Insomnia", "Sleep Apnea"]

plt.figure(figsize=(10, 6))
ax = sns.countplot(
    data=df, x="BMI Category", hue="Sleep Disorder",
    order=urutan_bmi, hue_order=urutan_gangguan,
    palette={"None": "#55A868", "Insomnia": "#C44E52", "Sleep Apnea": "#4C72B0"}
)
for wadah in ax.containers:
    ax.bar_label(wadah, padding=2)

plt.title("Jumlah Responden Berdasarkan Kategori BMI dan Gangguan Tidur")
plt.xlabel("Kategori BMI")
plt.ylabel("Jumlah Responden")
plt.legend(title="Gangguan Tidur")
plt.tight_layout()
plt.show()

tabel = pd.crosstab(df["BMI Category"], df["Sleep Disorder"]) \
          .reindex(index=urutan_bmi, columns=urutan_gangguan)
print("Persentase gangguan tidur per kategori BMI (%):")
display((tabel.div(tabel.sum(axis=1), axis=0) * 100).round(1))

**Interpretasi:**
Pada kategori BMI **Normal**, sekitar 92,6% responden (200 dari 216) tidak memiliki gangguan tidur. Pada kategori **Overweight**, hanya 12,8% yang tidak memiliki gangguan, sedangkan 43,2% mengalami insomnia dan 43,9% mengalami sleep apnea. Seluruh responden kategori **Obese** (10 orang) memiliki gangguan tidur. Dalam dataset ini, kelebihan berat badan jauh lebih sering muncul bersama gangguan tidur. Perlu diingat bahwa jumlah responden Obese sangat kecil, sehingga persentasenya tidak dapat digeneralisasi, dan ini bukan diagnosis medis.

### 7.7 Line Chart — Tren Tidur dan Stres Berdasarkan Kelompok Usia

In [ ]:
ringkas_usia = (
    df.groupby("Kelompok Usia", observed=True)
      [["Sleep Duration", "Quality of Sleep", "Stress Level"]].mean()
)
jumlah_usia = df["Kelompok Usia"].value_counts().reindex(urutan_usia)

plt.figure(figsize=(11, 6))
plt.plot(ringkas_usia.index, ringkas_usia["Sleep Duration"],
         marker="o", linewidth=2, label="Durasi tidur (jam)")
plt.plot(ringkas_usia.index, ringkas_usia["Quality of Sleep"],
         marker="s", linewidth=2, label="Kualitas tidur (skor)")
plt.plot(ringkas_usia.index, ringkas_usia["Stress Level"],
         marker="^", linewidth=2, linestyle="--", label="Tingkat stres (skor)")

plt.xticks(range(len(urutan_usia)),
           [f"{u}\n(n={jumlah_usia[u]})" for u in urutan_usia])
plt.title("Rata-rata Durasi Tidur, Kualitas Tidur, dan Stres per Kelompok Usia")
plt.xlabel("Kelompok Usia (tahun)")
plt.ylabel("Nilai Rata-rata (jam / skor)")
plt.legend()
plt.tight_layout()
plt.show()

display(ringkas_usia.round(2))

**Interpretasi:**
Tren kelompok usia tidak naik atau turun secara mulus. Kelompok **46–50 tahun** memiliki durasi tidur terpendek (6,32 jam), kualitas tidur terendah (6,32), dan stres tertinggi (7,38). Sebaliknya, kelompok **51–59 tahun** memiliki durasi tidur terpanjang (8,02 jam), kualitas tidur tertinggi (8,70), dan stres terendah (3,58). Garis stres bergerak berlawanan arah dengan garis durasi dan kualitas tidur, sesuai dengan hasil heatmap. Lonjakan pada kelompok 51–59 tahun kemungkinan dipengaruhi komposisi pekerjaan responden di kelompok tersebut, bukan sekadar faktor usia.

## 8. Ringkasan Statistik

In [ ]:
print("=" * 60)
print("RINGKASAN HASIL DATA VISUALIZATION")
print("=" * 60)
print(f"Jumlah responden : {len(df)}")
print(f"Jumlah variabel  : {n_kolom_asli}")

print("\n1. DURASI TIDUR")
print(f"   Rata-rata : {df['Sleep Duration'].mean():.2f} jam")
print(f"   Median    : {df['Sleep Duration'].median():.2f} jam")
print(f"   < 7 jam   : {(df['Sleep Duration'] < 7).mean()*100:.1f}% responden")

print("\n2. KUALITAS TIDUR")
print(f"   Rata-rata : {df['Quality of Sleep'].mean():.2f}")
print(f"   Rentang   : {df['Quality of Sleep'].min()} - {df['Quality of Sleep'].max()}")

print("\n3. TINGKAT STRES")
print(f"   Rata-rata : {df['Stress Level'].mean():.2f}")

print("\n4. KORELASI UTAMA")
print(f"   Durasi vs Kualitas tidur : {df['Sleep Duration'].corr(df['Quality of Sleep']):.3f}")
print(f"   Stres vs Kualitas tidur  : {df['Stress Level'].corr(df['Quality of Sleep']):.3f}")
print(f"   Stres vs Durasi tidur    : {df['Stress Level'].corr(df['Sleep Duration']):.3f}")

print("\n5. GANGGUAN TIDUR")
for nama, jml in df["Sleep Disorder"].value_counts().items():
    print(f"   {nama:<12}: {jml} ({jml/len(df)*100:.1f}%)")

print("\nCatatan: hasil ini adalah statistik deskriptif, bukan bukti")
print("hubungan sebab-akibat maupun diagnosis medis.")

## 9. Kesimpulan

1. **Karakteristik data.** Dataset terdiri dari 374 responden dan 13 variabel, tanpa nilai kosong. Perlu dicatat bahwa nilai `None` pada *Sleep Disorder* berarti tidak ada gangguan tidur, dan label BMI "Normal Weight" digabung dengan "Normal".
2. **Durasi tidur.** Rata-rata 7,13 jam, tetapi 41,4% responden tidur kurang dari 7 jam.
3. **Stres adalah faktor yang paling menonjol.** Stres berkorelasi negatif kuat dengan kualitas tidur (r = −0,90) dan durasi tidur (r = −0,81). Median durasi tidur turun dari 8,20 jam (stres rendah) menjadi 6,15 jam (stres tinggi).
4. **Durasi dan kualitas tidur saling berkaitan erat** (r = 0,88).
5. **Perbedaan antar kelompok.** Engineer memiliki kualitas tidur tertinggi (8,41), sedangkan Salesperson terendah (6,00). Kelompok usia 46–50 tahun menunjukkan kondisi tidur paling buruk dan stres tertinggi.
6. **Gangguan tidur dan BMI.** Gangguan tidur jauh lebih banyak ditemukan pada responden Overweight dan Obese dibandingkan responden dengan BMI Normal.

### Keterbatasan
- Banyak baris memiliki profil yang identik (242 baris duplikat tanpa `Person ID`), sehingga data tampak berpola berulang dan kemungkinan bukan sampel acak murni.
- Beberapa kelompok sangat kecil (misalnya Obese n = 10, Manager n = 1), sehingga tidak dapat digeneralisasi.
- Analisis bersifat deskriptif dan korelasional, sehingga tidak membuktikan sebab-akibat dan bukan diagnosis medis.

### Saran Pengembangan
Analisis dapat dilanjutkan dengan visualisasi interaktif menggunakan Plotly, analisis pengaruh tekanan darah, atau membangun model klasifikasi gangguan tidur sebagai tahap Data Mining berikutnya.